In [2]:
#ouvre les csv de Final_data / pic_activité 
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns

# Chemin vers le dossier contenant les fichiers CSV
folder_path = '../Final_data/pics_activité'
# Liste pour stocker les DataFrames
dataframes = []
# Parcourir tous les fichiers dans le dossier
for filename in os.listdir(folder_path):
    if filename.endswith('.csv'):
        file_path = os.path.join(folder_path, filename)
        print(f'Loading {filename}')


Loading bronchiolite-passages-aux-urgences-et-actes-sos-medecins-departement.csv
Loading covid-19-passages-aux-urgences-et-actes-sos-medecins-departement.csv
Loading grippe-passages-aux-urgences-et-actes-sos-medecins-departement.csv
Loading infections-respiratoires-aigues-ira-passages-aux-urgences-et-actes-sos-medecins-departement.csv


In [9]:
import pandas as pd

# Chemin du fichier
INPUT = "../Final_data/pics_activité/covid-19-passages-aux-urgences-et-actes-sos-medecins-departement.csv"
OUTPUT = "covid_prets_powerbi.csv"

df = pd.read_csv(INPUT)

# 1) Convertir la colonne date
df["date_semaine"] = pd.to_datetime(df["1er jour de la semaine"], dayfirst=True, errors="coerce")

# 2) Nettoyer la colonne "Semaine" -> extraire le numéro de semaine
# Gère les formats possibles: "2025-S39", "S39", "39", "2025-39", etc.
df["semaine_num"] = (
    df["Semaine"]
    .astype(str)
    .str.extract(r"(\d{1,2})$")[0]   # prend les 1-2 derniers chiffres
    .astype("Int64")
)

# 3) Créer la saison (juillet -> juin)
def build_saison(d):
    if pd.isna(d):
        return None
    y = d.year
    return f"{y}-{y+1}" if d.month >= 7 else f"{y-1}-{y}"

df["saison"] = df["date_semaine"].apply(build_saison)

# 4) Renommer les colonnes pour éviter les accents (Power BI aime bien)
df_out = df.rename(columns={
    "Département Code": "dep_code",
    "Département": "dep",
    "Classe d'âge": "age_classe",
    "Région Code": "region_code",
    "Région": "region",
    "Taux de passages aux urgences pour COVID-19": "taux_passages_urgences_covid",
    "Taux d'hospitalisations après passages aux urgences pour COVID-19": "taux_hosp_apres_urgences_covid",
    "Taux d'actes médicaux SOS médecins pour COVID-19": "taux_actes_sos_covid",
})

# 5) Garder un set de colonnes clean
cols = [
    "region_code", "region",
    "dep_code", "dep",
    "age_classe",
    "saison", "semaine_num", "date_semaine",
    "taux_passages_urgences_covid",
    "taux_hosp_apres_urgences_covid",
    "taux_actes_sos_covid",
]
df_out = df_out[cols]

# 6) Sauvegarder
df_out.to_csv(OUTPUT, index=False, encoding="utf-8")

print("OK ✅ CSV créé:", OUTPUT)
print(df_out.head())


C:\Users\cantfly\AppData\Local\Temp\ipykernel_17188\1432965591.py:10: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df["date_semaine"] = pd.to_datetime(df["1er jour de la semaine"], dayfirst=True, errors="coerce")


OK ✅ CSV créé: covid_prets_powerbi.csv
   region_code               region dep_code   dep      age_classe     saison  \
0           24  Centre-Val de Loire       18  Cher       05-14 ans  2023-2024   
1           24  Centre-Val de Loire       18  Cher       05-14 ans  2023-2024   
2           24  Centre-Val de Loire       18  Cher       15-64 ans  2023-2024   
3           24  Centre-Val de Loire       18  Cher  65 ans ou plus  2023-2024   
4           24  Centre-Val de Loire       18  Cher       Tous âges  2023-2024   

   semaine_num date_semaine  taux_passages_urgences_covid  \
0           16   2024-04-15                      0.000000   
1           17   2024-04-22                      0.000000   
2           17   2024-04-22                    233.644860   
3           18   2024-04-29                      0.000000   
4           18   2024-04-29                     65.231572   

   taux_hosp_apres_urgences_covid  taux_actes_sos_covid  
0                             0.0                

In [4]:
print(df_out.dtypes)

region_code                                int64
region                                    object
dep_code                                  object
dep                                       object
age_classe                                object
saison                                    object
semaine_num                                Int64
date_semaine                      datetime64[ns]
taux_passages_urgences_covid             float64
taux_hosp_apres_urgences_covid           float64
taux_actes_sos_covid                     float64
dtype: object


In [ ]:
# Modèle d'entraînement national pour prédire les trois indicateurs COVID sur 12 semaines
import numpy as np
from datetime import timedelta
from typing import Tuple

try:
    from sklearn.linear_model import LinearRegression
    from sklearn.metrics import mean_absolute_error
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "scikit-learn est requis pour cette cellule. Installez-le via 'pip install scikit-learn' puis relancez."
    ) from exc

TARGET_COLS = [
    "taux_passages_urgences_covid",
    "taux_hosp_apres_urgences_covid",
    "taux_actes_sos_covid",
]

weekly_series = (
    df_out
    .set_index("date_semaine")[TARGET_COLS]
    .sort_index()
    .resample("W-MON")
    .mean()
    .interpolate(method="linear")
)

lags = 12
horizon = 12

if len(weekly_series) < lags + horizon + 1:
    raise ValueError("Pas assez d'historique pour entraîner le modèle avec ces paramètres.")


def build_supervised_matrix(data: pd.DataFrame, n_lags: int, n_horizon: int) -> Tuple[np.ndarray, np.ndarray]:
    values = data.values
    X, y = [], []
    for idx in range(n_lags, len(values) - n_horizon):
        X.append(values[idx - n_lags:idx].flatten())
        y.append(values[idx:idx + n_horizon].flatten())
    return np.asarray(X), np.asarray(y)


X, y = build_supervised_matrix(weekly_series, lags, horizon)
if X.shape[0] < 2:
    raise ValueError("Impossible de constituer un jeu d'entraînement et de test. Réduisez lags ou horizon.")

split_index = X.shape[0] - 1
X_train, y_train = X[:split_index], y[:split_index]
X_test, y_test = X[split_index:], y[split_index:]

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
actual_matrix = y_test.reshape(horizon, len(TARGET_COLS))
pred_matrix = y_pred.reshape(horizon, len(TARGET_COLS))

mae_scores = {
    col: mean_absolute_error(actual_matrix[:, idx], pred_matrix[:, idx])
    for idx, col in enumerate(TARGET_COLS)
}
mae_df = pd.DataFrame.from_dict(mae_scores, orient="index", columns=["MAE_12_semaines"])

history_df = weekly_series.reset_index()
history_df["annee"] = history_df["date_semaine"].dt.year
history_df["mois"] = history_df["date_semaine"].dt.month
history_df["mois_nom"] = history_df["date_semaine"].dt.month_name()
history_df = history_df[[
    "date_semaine", "annee", "mois", "mois_nom",
    "taux_passages_urgences_covid",
    "taux_hosp_apres_urgences_covid",
    "taux_actes_sos_covid",
]]

model_full = LinearRegression()
model_full.fit(X, y)
latest_window = weekly_series.values[-lags:].flatten().reshape(1, -1)
forecast = model_full.predict(latest_window).reshape(horizon, len(TARGET_COLS))
forecast_index = pd.date_range(start=weekly_series.index[-1] + timedelta(weeks=1), periods=horizon, freq="W-MON")
forecast_df = pd.DataFrame(forecast, index=forecast_index, columns=TARGET_COLS).reset_index()
forecast_df = forecast_df.rename(columns={"index": "date_semaine"})
forecast_df["annee"] = forecast_df["date_semaine"].dt.year
forecast_df["mois"] = forecast_df["date_semaine"].dt.month
forecast_df["mois_nom"] = forecast_df["date_semaine"].dt.month_name()
forecast_df = forecast_df[[
    "date_semaine", "annee", "mois", "mois_nom",
    "taux_passages_urgences_covid",
    "taux_hosp_apres_urgences_covid",
    "taux_actes_sos_covid",
]]

mae_df, history_df.tail(), forecast_df.head()

TypeError: 'type' object is not subscriptable

In [ ]:
# Enregistrer les séries agrégées pour Power BI (une ligne par semaine)
history_df.to_csv("covid_history_national.csv", index=False, encoding="utf-8")
forecast_df.to_csv("covid_forecast_12_semaines.csv", index=False, encoding="utf-8")

print("Fichiers enregistrés :")
print(" - covid_history_national.csv")
print(" - covid_forecast_12_semaines.csv")